In [1]:
pip install speechbrain


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 788.2/788.2 kB 16.6 MB/s eta 0:00:00
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
  Attempting uninstall: ruamel.yaml
    Found existing installation: ruamel.yaml 0.19.1
    Uninstalling ruamel.yaml-0.19.1:
      Successfully uninstalled ruamel.yaml-0.19.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, b

In [2]:
import os
import torch
import torchaudio
from speechbrain.inference.separation import SepformerSeparation

In [3]:
# Re-declare the exact network wrapper architecture from your notebook
class UnifiedSepFormer(torch.nn.Module):
    def __init__(self, modules_dict):
        super().__init__()
        self.encoder = modules_dict['encoder']
        self.masknet = modules_dict['masknet']
        self.decoder = modules_dict['decoder']
        
    def forward(self, mix):
        mix_w = self.encoder(mix)
        est_mask = self.masknet(mix_w)
        
        decoded_sources = []
        num_sources = est_mask.shape[0]
        
        for i in range(num_sources):
            sep_h_i = mix_w * est_mask[i]
            est_source_i = self.decoder(sep_h_i)
            decoded_sources.append(est_source_i.unsqueeze(-1))
            
        est_source = torch.cat(decoded_sources, dim=-1)
        return est_source

In [4]:
def main():
    checkpoint_path = "/kaggle/input/notebooks/liamgraphics/new-3p-speechbrain/model_checkpoints/best_audio_separation_model.pt"
    save_directory = "./tmp_sepformer_model"
    output_quantized_path = "./model_checkpoints/quantized_audio_separation_model.pt"
    
    # 1. Verification check
    if not os.path.exists(checkpoint_path):
        print(f"[-] Error: Checkpoint file not found at {checkpoint_path}.")
        print("    Please ensure your training run has completed and saved files successfully.")
        return

    # 2. Instantiate blueprint structural layers on CPU for quantization configuration
    print("[+] Loading SepFormer structural submodules from Hug...")
    model_hub = SepformerSeparation.from_hparams(
        source="speechbrain/sepformer-libri3mix", 
        savedir=save_directory,
        run_opts={"device": "cpu"} # Quantization conversions must execute on CPU targets
    )
    
    # Initialize full model wrapper shell
    model = UnifiedSepFormer(model_hub.mods)
    
    # 3. Load trained model state weights dictionary
    print(f"[+] Loading trained FP32 parameters from {checkpoint_path}...")
    checkpoint = torch.load(checkpoint_path, map_location="cpu")
    model.load_state_dict(checkpoint['model_state_dict'])
    
    # Put model into evaluation mode before configuration changes
    model.eval()
    
    # Compute uncompressed parameter footprint for analysis
    torch.save(model.state_dict(), "temp_uncompressed.pt")
    orig_size_mb = os.path.getsize("temp_uncompressed.pt") / (1024 * 1024)
    os.remove("temp_uncompressed.pt")
    print(f"[*] Original FP32 Model Weight Footprint: {orig_size_mb:.2f} MB")
    
    # 4. Trigger Dynamic Quantization engine targeting Linear projection modules
    print("[+] Applying Post-Training Dynamic Quantization (qint8) to Linear layers...")
    quantized_model = torch.quantization.quantize_dynamic(
        model, 
        qconfig_spec={torch.nn.Linear}, 
        dtype=torch.qint8
    )

    os.makedirs(os.path.dirname(output_quantized_path), exist_ok=True)
    
    # 5. Export lightweight compressed weights
    print(f"[+] Saving optimized INT8 model weights to: {output_quantized_path}")
    torch.save(quantized_model.state_dict(), output_quantized_path)
    
    quant_size_mb = os.path.getsize(output_quantized_path) / (1024 * 1024)
    print(f"[*] Quantized INT8 Model Weight Footprint: {quant_size_mb:.2f} MB")
    print(f"[SUCCESS] Model compressed successfully! ~{orig_size_mb / quant_size_mb:.2f}x smaller.")

if __name__ == "__main__":
    main()

[+] Loading SepFormer structural submodules from Hug...


hyperparams.yaml: 0.00B [00:00, ?B/s]

encoder.ckpt:   0%|          | 0.00/17.3k [00:00<?, ?B/s]

masknet.ckpt:   0%|          | 0.00/113M [00:00<?, ?B/s]

decoder.ckpt:   0%|          | 0.00/17.3k [00:00<?, ?B/s]

[+] Loading trained FP32 parameters from /kaggle/input/notebooks/liamgraphics/new-3p-speechbrain/model_checkpoints/best_audio_separation_model.pt...
[*] Original FP32 Model Weight Footprint: 108.15 MB
[+] Applying Post-Training Dynamic Quantization (qint8) to Linear layers...


/tmp/ipykernel_22/1091964485.py:39: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = torch.quantization.quantize_dynamic(


[+] Saving optimized INT8 model weights to: ./model_checkpoints/quantized_audio_separation_model.pt
[*] Quantized INT8 Model Weight Footprint: 60.21 MB
[SUCCESS] Model compressed successfully! ~1.80x smaller.
